# Socioeconomic Development and Environmental Performance

## Research Question

**How is socioeconomic development associated with environmental performance across countries?**

## Research Questions

1. Is economic prosperity associated with environmental performance?
2. Do countries at different economic levels show different environmental performance?
3. Do countries with higher educational participation have higher environmental performance?
4. How is urbanization associated with environmental performance?
5. Does the relationship between urbanization and environmental performance differ across environmental dimensions such as air quality, waste management, and environmental health?
6. Are employment and unemployment rates associated with environmental performance?
7. Which countries perform substantially better or worse environmentally than their economic level might suggest?
8. Which socioeconomic indicator shows the strongest relationship with environmental performance?

## 1. Setup

Importing the libraries required for data manipulation and visualization.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Data Loading and Initial Inspection

### 2.1 Environmental Performance Index (EPI)

The EPI dataset is loaded and initially inspected to understand its structure, dimensions, and available environmental indicators.

In [9]:
epi = pd.read_excel("data/epi/epi2026results2026-07-07.xlsx", sheet_name="data")

In [11]:
epi.head(5)

,iso,country,EPI.new,HLT.new,AIR.new,PMD.new,HFD.new,OZD.new,NOD.new,COE.new,...,GRP.6,GRP.7,GRP.8,GRP.9,region,groupname,GDP,GPC,POP,LDA
0,AFG,Afghanistan,33.14,24.66,24.44,40.18,4.86,20.28,65.61,49.72,...,SWZ,TKM,UGA,CAF,Southern Asia,Landlocked Developing Countries,83.677383,1962.070438,42.647492,652230.0
1,ALB,Albania,49.22,48.67,37.37,32.07,38.77,60.79,60.26,62.46,...,SRB,ROU,HUN,BIH,Eastern Europe,Central European Initiative,51.443601,18950.592789,2.714617,27400.0
2,DZA,Algeria,42.74,50.96,54.83,14.37,100.00,29.89,47.85,44.50,...,SAU,EGY,OMN,ARE,Greater Middle East,Arab League,725.711643,15501.919700,46.814308,2381740.0
3,AGO,Angola,39.46,24.93,19.45,18.77,17.19,26.62,35.63,38.66,...,KIR,BFA,UGA,NPL,Sub-Saharan Africa,Least Developed Countries,337.255561,8901.887366,37.885849,1246700.0
4,ATG,Antigua & Barbuda,48.04,66.07,68.52,54.22,80.05,100.00,34.12,82.50,...,BHS,KNA,SUR,LCA,Latin America & Caribbean,Small Island Developing States,2.754237,29371.637779,0.093772,440.0


In [6]:
epi.columns

Index(['iso', 'country', 'EPI.new', 'HLT.new', 'AIR.new', 'PMD.new', 'HFD.new',
       'OZD.new', 'NOD.new', 'COE.new',
       ...
       'GRP.6', 'GRP.7', 'GRP.8', 'GRP.9', 'region', 'groupname', 'GDP', 'GPC',
       'POP', 'LDA'],
      dtype='str', length=422)

### 2.2 Selecting Relevant EPI Variables

The original EPI dataset contains 422 columns. For the initial analysis, variables related to overall environmental performance, environmental health, air quality, waste management, and geographical region are selected.

The original dataset is preserved, while the selected variables are stored separately for analysis.

In [5]:
relevant_columns = [
    "iso",
    "country",
    "EPI.new",
    "HLT.new",
    "AIR.new",
    "WMG.new",
    "region"
]

epi_analysis = epi[relevant_columns].copy()

In [6]:
epi_analysis.head()

,iso,country,EPI.new,HLT.new,AIR.new,WMG.new,region
0,AFG,Afghanistan,33.14,24.66,24.44,56.20,Southern Asia
1,ALB,Albania,49.22,48.67,37.37,91.69,Eastern Europe
2,DZA,Algeria,42.74,50.96,54.83,40.00,Greater Middle East
3,AGO,Angola,39.46,24.93,19.45,43.89,Sub-Saharan Africa
4,ATG,Antigua & Barbuda,48.04,66.07,68.52,49.50,Latin America & Caribbean


### 2.3 Data Quality Check

The selected dataset is examined for its dimensions, data types, missing values, duplicate rows, and country coverage before further analysis.

In [10]:
epi_analysis.shape

(177, 7)

In [8]:
epi_analysis.info()

<class 'pandas.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   iso      177 non-null    str    
 1   country  177 non-null    str    
 2   EPI.new  177 non-null    float64
 3   HLT.new  177 non-null    float64
 4   AIR.new  177 non-null    float64
 5   WMG.new  177 non-null    float64
 6   region   177 non-null    str    
dtypes: float64(4), str(3)
memory usage: 9.8 KB


In [14]:
print("Duplicate rows:", epi_analysis.duplicated().sum())
print("Unique countries:", epi_analysis["country"].nunique())

Duplicate rows: 0
Unique countries: 177


## 3. Socioeconomic Data

### 3.1 GDP per Capita

GDP per capita data from the World Bank is loaded as the first socioeconomic indicator. The first four metadata rows are skipped so that the dataset is read using its actual column headers.

In [12]:
gdp = pd.read_csv("data/world_bank/gdp/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_33610.csv", skiprows=4)

In [13]:
gdp.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30654.485124,22664.370995,26827.344787,31000.571380,34897.618393,38590.565029,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089515,186.909365,197.367876,225.400456,208.963066,226.836513,...,1529.923080,1553.617944,1508.031525,1351.503167,1560.894626,1675.902524,1571.132704,1628.227289,1722.385620,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,416.871146,NaN,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.551510,155.587905,...,1577.203497,1723.114463,2219.412555,2034.437940,2116.938294,2143.072094,1846.246811,1416.228412,1600.058374,NaN
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2832.149980,2891.830324,2507.868072,1749.179484,2266.968349,3598.536691,2885.513491,2720.819007,3129.476623,NaN


#### Selecting the Reference Year

The most recent years are examined to determine which provides better data coverage for the analysis.

In [15]:
gdp['2024'].isna().sum()

np.int64(18)

In [16]:
gdp['2025'].isna().sum()

np.int64(32)

The 2024 column contains 18 missing values, compared with 32 in 2025. Therefore, 2024 is selected as the reference year because it provides more complete data coverage.

#### Selecting Relevant Variables

Only the country name, country code, and 2024 GDP per capita values are retained for the analysis. The country code will later be used to match the World Bank data with the EPI dataset.

In [17]:
relevant_gdp_columns = ['Country Name', 'Country Code', '2024']
gdp_analysis = gdp[relevant_gdp_columns].copy()

In [18]:
gdp_analysis.head()

,Country Name,Country Code,2024
0,Aruba,ABW,38590.565029
1,Africa Eastern and Southern,AFE,1628.227289
2,Afghanistan,AFG,416.871146
3,Africa Western and Central,AFW,1416.228412
4,Angola,AGO,2720.819007


#### Matching Countries with the EPI Dataset

World Bank country codes are compared with the ISO codes in the EPI dataset. This ensures that the GDP dataset contains only countries relevant to the environmental analysis and removes aggregate World Bank entries that do not represent individual EPI countries.

In [22]:
mask = gdp_analysis["Country Code"].isin(epi_analysis["iso"])
gdp_matched = gdp_analysis[mask]
gdp_matched

,Country Name,Country Code,2024
2,Afghanistan,AFG,416.871146
4,Angola,AGO,2720.819007
5,Albania,ALB,11374.008578
8,United Arab Emirates,ARE,50273.512624
9,Argentina,ARG,13969.783660
...,...,...,...
257,Vanuatu,VUT,3959.877035
259,Samoa,WSM,5392.877619
262,South Africa,ZAF,6267.186814
263,Zambia,ZMB,1187.109434


In [20]:
epi_analysis[~epi_analysis['iso'].isin(gdp['Country Code'])]

,iso,country,EPI.new,HLT.new,AIR.new,WMG.new,region
155,TWN,Taiwan,46.02,60.11,48.86,97.88,Asia-Pacific


In [23]:
gdp_matched.isna().sum()

Country Name    0
Country Code    0
2024            0
dtype: int64

**Matching result:** 176 of the 177 EPI countries were successfully matched with the World Bank GDP data. Taiwan (TWN) was the only unmatched country. Among the 176 matched countries, no missing values were found for 2024 GDP per capita.

In [24]:
gdp_matched.rename(columns = {'2024': 'GDP_per_capita'}, inplace=True)

In [25]:
gdp_matched.columns

Index(['Country Name', 'Country Code', 'GDP_per_capita'], dtype='str')

In [26]:
gdp_matched.shape

(176, 3)

**GDP preparation result:** The final GDP dataset contains 176 matched countries and 3 variables. The 2024 GDP per capita values are complete for all matched countries, while Taiwan (TWN) remains unmatched with the World Bank data.